# Part A
**Members**: Daniel Andrade - Andrés Vega - Carlos Flores

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "seclab").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

### 1. SQLi

**OWASP 2021 category**: A03:2021 – Injection

In [2]:
from homework.hw2.andrade.attack.attack import confirm_sqli

findings = confirm_sqli()
attacks = [f for f in findings if f.payload.family == "sqli"]
benign  = [f for f in findings if f.payload.family == "benign"]

assert attacks and all(f.confirmed for f in attacks), "every SQLi payload must leak the canary"
assert benign and not any(f.confirmed for f in benign), "benign login must NOT confirm"

print(f"SQLi: {len(attacks)}/{len(attacks)} payloads leaked the canary (extraction oracle)")
print(f"benign control confirmed: {any(f.confirmed for f in benign)}  (the oracle is not giving a false alarm)")

SQLi: 3/3 payloads leaked the canary (extraction oracle)
benign control confirmed: False  (the oracle is not giving a false alarm)


### 2. XSS

**OWASP 2021 category**: A03:2021 – Injection

In [3]:
from homework.hw2.andrade.attack.attack import confirm_xss

vuln_f, safe_f = confirm_xss()
assert vuln_f.confirmed, "reflected XSS on / must confirm (unescaped <script>)"
assert not safe_f.confirmed, "/safe html.escape()s the input — encoded reflection is NOT XSS"
print("vulnerable / :", "CONFIRMED" if vuln_f.confirmed else "no effect")
print("fixed /safe  :", "CONFIRMED" if safe_f.confirmed else "no effect", "(true negative)")

vulnerable / : CONFIRMED
fixed /safe  : no effect (true negative)


### 3. Command Injection

**OWASP 2021 category**: A03:2021 – Injection

In [4]:
from homework.hw2.andrade.attack.attack import confirm_cmdi

findings = confirm_cmdi()
attacks = [f for f in findings if f.payload.family == "cmdi"]
benign = [f for f in findings if f.payload.family == "benign"]
assert attacks and all(f.confirmed for f in attacks), (
    "metacharacter payloads must confirm"
)
assert benign and not any(f.confirmed for f in benign), "a plain host must NOT confirm"
print(
    f"cmdi: {len(attacks)}/{len(attacks)} confirmed; benign host confirmed: {any(f.confirmed for f in benign)}"
)

cmdi: 3/3 confirmed; benign host confirmed: False


Broken Access Control (A01:2021) is not reachable on vuln-web as written, because there is no access-control logic to break, i.e., no authenticated session, no per-user resource ownership check, no role gate. 

The admin secret leak is a consequence of Injection (A03), not of an access-control decision being bypassed. BAC would require a second resource (e.g. "view user N's profile") with a privilege check that a non-owner could evade; this app has no such resource.


In [5]:
from homework.hw2.andrade import webharness

webharness.down()